# 第 06 课：模块、程序入口与小型项目组织

**前置知识：** 函数与返回值、字典与循环、Path/JSON 文件读写，以及上一课的异常检查。  
**建议用时：** 2–3 小时；练习可再安排 1–2 小时。  
**配套脚本：** `lesson_06_modules_and_main.py`，可独立运行。

今天把熟悉的字符编码程序组织成可以导入、可以单独启动的小项目。核心问题是：别的文件想复用一个函数时，怎样避免把示例处理也一起启动？

你将掌握：import 的行为、模块名与函数名、main 普通函数、入口判断，以及工作目录与脚本目录的区别。

本课保留少量已学的输入检查，使用固定的三条示例观察模块行为。完整的文件数据校验流程仍可参考上一课。本课无需 GPU 或模型权重。


## 使用方式与文件约定

按顺序运行分步示例。“完整程序”单元与配套 .py 完全一致，可以独立执行；三个参考答案也各自带齐依赖。练习起始单元只有合法注释，不会阻断运行全部，但仍需自己完成练习。

本课会生成以下文件，路径均会打印：

| 位置 | 用途 | 重复运行的行为 |
| --- | --- | --- |
| `lesson_06_data/generated_records.json` | 独立完整程序的结果，相对于当前工作目录 | 重新生成 |
| `lesson_06_workspace/generated_demos/` | 观察导入与入口的短示例 | 按讲义模板重建；修改练习请用下方 project |
| `lesson_06_workspace/project/text_tools.py` | 供你编辑的工具模块 | 只在文件缺失时创建，保留已有修改 |
| `lesson_06_workspace/project/app.py` | 供你编辑的项目入口 | 只在文件缺失时创建，保留已有修改 |
| `project/generated_data/generated_records.json` | 两文件项目的生成结果，位于 project 内 | 直接运行 app 或明确调用 main 时更新 |
| `lesson_06_workspace/generated_answer_a/`、`generated_answer_b/`、`generated_answer_c/` | 参考答案专用演示 | 执行对应答案时重建 |

不要把个人笔记放进标记为“生成”的文件。已保存的输出来自验证环境；本地重跑后路径会变化。


## 0. 准备运行辅助

为了观察真正的 .py 文件行为，讲义用下面的 run_python 作为“运行按钮”：每次启动一个新的 Python 进程，并指定它的工作目录。

本课先把这段辅助代码运行一次即可，不要求记忆 subprocess 的参数。这里没有修改 Notebook 的模块搜索路径，也不会沿用它之前导入的模块。

- `sys.executable`：当前使用的 Python 解释器。
- `-X utf8`：让这些演示进程按 UTF-8 处理文字，便于显示中文。
- `cwd`：给演示进程指定工作目录。
- “退出码 0”：这个演示进程正常结束；它本身不能证明所有业务结果都正确。

调用 `run_python(["app.py"], 某个目录)` 相当于在那个目录启动 `python app.py`。`["-c", "Python代码"]` 则直接执行一小段代码，无需另建探测脚本。


In [1]:
import subprocess
import sys
from pathlib import Path


def run_python(arguments, directory):
    # 用当前解释器启动一个新进程，避免沿用 Notebook 的模块缓存。
    command = [sys.executable, "-X", "utf8"] + arguments
    completed = subprocess.run(
        command,
        cwd=directory,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    print("退出码：", completed.returncode)
    if completed.returncode != 0:
        raise RuntimeError("示例进程执行失败，请查看上面的错误。")


def write_if_missing(path, source):
    if path.exists():
        print("保留已有文件：", path.resolve())
    else:
        path.write_text(source, encoding="utf-8")
        print("已创建：", path.resolve())

workspace_dir = Path.cwd() / "lesson_06_workspace"
demo_dir = workspace_dir / "generated_demos"
project_dir = workspace_dir / "project"
demo_dir.mkdir(parents=True, exist_ok=True)
project_dir.mkdir(parents=True, exist_ok=True)
print("练习项目目录：", project_dir.resolve())


练习项目目录： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\project


## 1. 模块就是保存定义与语句的 .py 文件

前几课的 `import json`，就是让程序使用 json 模块。本课开始导入自己写的模块。

我们先把一个短函数写入 lesson06_probe.py。下面三单引号之间的内容是一个**多行字符串**；write_text 把这段文字保存成 .py 文件，这一步只是写文件，还没有运行里面的代码。

本例故意在函数外放一句 print，用来观察导入时发生了什么。实际可复用工具模块通常避免这类无关输出。


In [2]:
probe_source = '''print("执行了工具模块的顶层语句")


def add_label(text):
    return "原文：" + text
'''

probe_path = demo_dir / "lesson06_probe.py"
probe_path.write_text(probe_source, encoding="utf-8")
print("示例模块：", probe_path.resolve())


示例模块： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\generated_demos\lesson06_probe.py


## 2. import 会执行模块顶层语句

**先预测：** 同一个 Python 进程连续两次 import，模块顶层的 print 会出现几次？函数 add_label 会不会自动被调用？

`lesson06_probe.add_label(...)` 的点号表示“使用 lesson06_probe 模块里的 add_label 函数”。文件后缀 .py 不写在 import 后面。


In [3]:
probe_code = '''print("第一次导入")
import lesson06_probe
print("第二次导入")
import lesson06_probe
print("现在主动调用函数")
print(lesson06_probe.add_label("我爱学习"))
'''

run_python(["-c", probe_code], demo_dir)


第一次导入
执行了工具模块的顶层语句
第二次导入
现在主动调用函数
原文：我爱学习
退出码： 0


在正常导入流程中，同一进程第一次导入该模块会执行其顶层语句，之后通常复用已经导入的模块，因此顶层提示只出现一次。def 会建立函数定义；函数体要等到调用时才执行。

还有一种写法：`from lesson06_probe import add_label`，之后可以直接调用 add_label。两种写法都在使用同一模块的函数，本课项目统一采用 `import text_tools`，方便看清函数来自哪里。

如果在同一个 Notebook 内核中导入模块后再改文件，再次 import 通常不会自动加载修改。本课每次启动新进程；以后在内核里直接做导入实验时，可以重启内核后按顺序重跑。


## 3. main() 不会因为名字叫 main 就自动运行

main 只是一个普通函数名，常用来集中组织程序启动后的主要步骤。真正决定要不要调用它的是这一段：

```python
if __name__ == "__main__":
    main()
```

- `__name__` 是 Python 为执行环境设置的名字，两侧各有两个下划线。
- 直接用 `python 文件.py` 启动时，这个模块中的 __name__ 值为字符串 `"__main__"`，磁盘文件名不变。
- 用 import 导入时，名字通常是模块名；当前简单例子是去掉 .py 的文件名。
- `==` 比较两个值是否相等。

先创建一个带入口判断的小文件，分别观察两种启动方式。


In [4]:
entry_source = '''print("顶层 __name__：", __name__)


def main():
    print("main 被调用了")


if __name__ == "__main__":
    main()
'''

entry_path = demo_dir / "entry_demo.py"
entry_path.write_text(entry_source, encoding="utf-8")
print("直接运行：")
run_python(["entry_demo.py"], demo_dir)


直接运行：
顶层 __name__： __main__
main 被调用了
退出码： 0


In [5]:
print("由其他代码导入：")
run_python(["-c", "import entry_demo; print('调用方：导入完成')"], demo_dir)


由其他代码导入：
顶层 __name__： entry_demo
调用方：导入完成
退出码： 0


比较结果：直接运行时，顶层 __name__ 为 __main__，随后调用 main；导入时，顶层 __name__ 为 entry_demo，跳过入口条件里的调用。

**入口判断只控制缩进在它下面的代码。** 文件外层的 print 仍然执行，所以不能把这个规则记成“import 不执行文件”。

因此，我们把可复用函数留在模块顶层，把创建目录、处理示例、保存结果等启动行为放进 main，最后用入口判断调用 main。


## 4. 独立完整程序：先把启动步骤收进 main

下一单元与配套 .py 完全一致。encode 与 decode 沿用已有思路，词表保持不变。

main 接收 output_dir 这个 Path 参数，在函数内准备词表、处理三条固定文本、写出 JSON，并返回记录列表。main 可以有参数，也可以有返回值；它仍是普通函数。

底部入口判断只负责选择输出位置并调用 main。导入这个文件时不会处理示例或写出这份 JSON；调用者如果明确调用 main，才会运行这些步骤。

普通 Jupyter 内核中的 __name__ 通常也是 __main__，所以本单元会运行示例。要观察“被 import”的情况，请使用前后的真实文件实验。

当前词表由示例代码保证有效：token_to_id 含有 <unk>，且与 id_to_token 对应。函数检查主要输入的类型和编号范围，没有重新实现上一课的整份 JSON 结构检查。


In [7]:
# 第 06 课：模块、程序入口与小型项目组织。
# 运行方法：python lesson_06_modules_and_main.py
# 只用标准库；独立运行，不依赖讲义生成的两文件项目。
# 直接运行：处理固定示例，更新当前工作目录下 lesson_06_data/generated_records.json。
# 导入本文件：定义函数，不调用 main，不生成上述数据文件。
# 词表沿用前几课的约定；本课重点是组织与调用代码。

import json
from pathlib import Path


def encode(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


def decode(token_ids, id_to_token):
    if not isinstance(token_ids, list):
        raise ValueError("token_ids 必须是列表")

    restored = ""
    for token_id in token_ids:
        if isinstance(token_id, bool):
            raise ValueError("编号不能是布尔值")
        if not isinstance(token_id, int):
            raise ValueError("编号必须是整数")
        if token_id < 0:
            raise ValueError("编号不能小于 0")
        if token_id >= len(id_to_token):
            raise ValueError("编号超出词表范围")
        restored = restored + id_to_token[token_id]
    return restored


def main(output_dir):
    token_to_id = {
        "我": 0, "爱": 1, "学": 2, "习": 3,
        "模": 4, "型": 5, "<unk>": 6,
    }
    id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

    texts = ["我爱学习", "我爱大模型", ""]
    records = []
    for text in texts:
        token_ids = encode(text, token_to_id)
        restored = decode(token_ids, id_to_token)
        record = {
            "text": text,
            "token_ids": token_ids,
            "decoded_text": restored,
        }
        records.append(record)

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "generated_records.json"
    json_text = json.dumps(records, ensure_ascii=False, indent=2)
    output_path.write_text(json_text, encoding="utf-8")

    print("记录数量：", len(records))
    print("结果文件：", output_path.resolve())
    for record in records:
        print(record)
    return records


if __name__ == "__main__":
    main(Path.cwd() / "lesson_06_data")


记录数量： 3
结果文件： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_data\generated_records.json
{'text': '我爱学习', 'token_ids': [0, 1, 2, 3], 'decoded_text': '我爱学习'}
{'text': '我爱大模型', 'token_ids': [0, 1, 6, 4, 5], 'decoded_text': '我爱<unk>模型'}
{'text': '', 'token_ids': [], 'decoded_text': ''}


### 观察 main 的边界

输出目录通过参数传入；词表、texts、records 等过程变量在 main 内部。函数返回的是 Python 记录列表，不会自动变成终端退出码。

如果在别的程序里导入配套脚本，可以只调用 encode/decode，也可以明确调用 main 并传入另一条输出路径。模块搜索位置需要正确，下一节先在同一目录的两个文件之间练习。

默认三条输出对应“我爱学习”“我爱<unk>模型”和空字符串；原文仍单独保存在 text 字段。


## 5. 拆成两个文件，各自承担清楚的职责

| 文件 | 放什么 | 怎样使用 |
| --- | --- | --- |
| text_tools.py | encode、decode 的定义 | 被其他程序导入和调用 |
| app.py | 导入工具、准备输入、调用函数、保存结果、main 入口 | 直接运行，或导入后明确调用 main |

两个文件放在同一 project 目录。本课直接运行 app.py 时，Python 会把入口脚本所在目录加入模块搜索位置，因此可以找到旁边的 text_tools.py。

下面创建的是另一种组织形式，与独立配套脚本生成相同的三条记录。它不需要导入配套脚本，也不依赖前几课的文件。

**仅缺失时创建：** 你编辑过的 text_tools.py 与 app.py 会保留。再次运行模板单元不会用模板覆盖已有文件；要做变化题，请编辑实际生成的 .py 并保存。


In [8]:
tools_source = '''# 可复用的字符编码工具：导入时只定义函数。

def encode(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


def decode(token_ids, id_to_token):
    if not isinstance(token_ids, list):
        raise ValueError("token_ids 必须是列表")

    restored = ""
    for token_id in token_ids:
        if isinstance(token_id, bool):
            raise ValueError("编号不能是布尔值")
        if not isinstance(token_id, int):
            raise ValueError("编号必须是整数")
        if token_id < 0:
            raise ValueError("编号不能小于 0")
        if token_id >= len(id_to_token):
            raise ValueError("编号超出词表范围")
        restored = restored + id_to_token[token_id]
    return restored
'''

write_if_missing(project_dir / "text_tools.py", tools_source)


已创建： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\project\text_tools.py


## 6. 编写 app.py：用模块名调用函数

app.py 中的 `text_tools.encode(...)` 和 `text_tools.decode(...)` 明确表示调用工具模块里的函数。这里没有再复制它们的定义。

两文件项目将生成结果固定放在自己的目录内。新增的路径表达式逐层理解：

```python
project_dir = Path(__file__).resolve().parent
```

__file__ 是这个真实 .py 模块的文件路径；resolve() 得到解析后的绝对路径；parent 取它所在的目录。这个表达式放在 app.py 的 main 中执行。

**不要直接把 __file__ 当作 Notebook 的文件路径。** 普通 Notebook 代码单元通常没有这个变量。本课把含有 __file__ 的代码写成文件，再启动该文件。


In [9]:
app_source = '''# 从入口调用工具模块；生成结果放在本项目目录内。
import json
from pathlib import Path
import text_tools


def main():
    project_dir = Path(__file__).resolve().parent
    output_dir = project_dir / "generated_data"
    print("当前工作目录：", Path.cwd())
    print("脚本目录：", project_dir)
    token_to_id = {
        "我": 0, "爱": 1, "学": 2, "习": 3,
        "模": 4, "型": 5, "<unk>": 6,
    }
    id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

    texts = ["我爱学习", "我爱大模型", ""]
    records = []
    for text in texts:
        token_ids = text_tools.encode(text, token_to_id)
        restored = text_tools.decode(token_ids, id_to_token)
        record = {
            "text": text,
            "token_ids": token_ids,
            "decoded_text": restored,
        }
        records.append(record)

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "generated_records.json"
    json_text = json.dumps(records, ensure_ascii=False, indent=2)
    output_path.write_text(json_text, encoding="utf-8")

    print("记录数量：", len(records))
    print("结果文件：", output_path.resolve())
    for record in records:
        print(record)
    return records


if __name__ == "__main__":
    main()
'''

write_if_missing(project_dir / "app.py", app_source)


已创建： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\project\app.py


## 7. 运行两文件项目

run_python 会在 project 目录启动新的 Python，运行 app.py。观察导入工具、编码、解码和保存结果是否形成完整流程。

若在终端操作：先进入程序打印的 project 目录，再执行 `python app.py`。用你本机一直可用的 python / python3 / py 命令即可。


In [10]:
run_python(["app.py"], project_dir)


当前工作目录： d:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\project
脚本目录： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\project
记录数量： 3
结果文件： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\project\generated_data\generated_records.json
{'text': '我爱学习', 'token_ids': [0, 1, 2, 3], 'decoded_text': '我爱学习'}
{'text': '我爱大模型', 'token_ids': [0, 1, 6, 4, 5], 'decoded_text': '我爱<unk>模型'}
{'text': '', 'token_ids': [], 'decoded_text': ''}
退出码： 0


再启动一个新进程，只导入 app，并由调用方打印“导入完成”。这次不应出现记录数量、输出路径和三条记录，因为 app 的 main 没有被调用。

导入过程中可能出现 Python 自动生成的 __pycache__，这是字节码缓存，不是 main 生成的业务结果文件。


In [11]:
run_python(["-c", "import app; print('导入完成，没有主动调用 main')"], project_dir)


导入完成，没有主动调用 main
退出码： 0


## 8. 工作目录与脚本目录是两个位置

| 表达式 | 表示什么 | 换启动目录后 |
| --- | --- | --- |
| `Path.cwd()` | 进程当前的工作目录 | 可能变化 |
| `Path(__file__).resolve().parent` | 当前 .py 模块所在目录 | 对同一个脚本保持相同位置 |

模块搜索与数据文件路径也要分别考虑：app.py 能找到旁边的 text_tools.py，并不意味着所有相对数据路径也都自动以 app.py 为起点。

下面从 project 的上一级启动同一个 app.py。你会看到工作目录变化，脚本目录和结果位置保持不变。


In [12]:
run_python(["project/app.py"], workspace_dir)


当前工作目录： d:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace
脚本目录： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\project
记录数量： 3
结果文件： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\project\generated_data\generated_records.json
{'text': '我爱学习', 'token_ids': [0, 1, 2, 3], 'decoded_text': '我爱学习'}
{'text': '我爱大模型', 'token_ids': [0, 1, 6, 4, 5], 'decoded_text': '我爱<unk>模型'}
{'text': '', 'token_ids': [], 'decoded_text': ''}
退出码： 0


本课特意对比两种输出约定：独立完整程序由入口把“当前工作目录下的 lesson_06_data”传给 main；两文件项目在 app.main 内使用脚本目录下的 generated_data。二者应生成相同的三条记录，但文件位置遵循各自约定。

后续项目可以由配置或命令行参数选择路径。现在先做到：看代码能说清位置依据，并用打印出的绝对路径核对。

## 常见问题

- **ModuleNotFoundError：** 先核对两个 .py 是否在同一 project 目录、文件名是否拼对、是否确实启动了这个 app.py。配套完整脚本可以独立运行；两文件项目需要两个文件都在。
- **导入就打印或写文件：** 检查这些语句是否仍放在模块顶层；入口条件只控制自己下面的缩进块。
- **改了模块却没变化：** 保存实际的 .py 文件，再启动新进程；在已有内核中直接 import 时，留意模块缓存。
- **改了讲义模板但磁盘文件没变：** 这是仅缺失时创建的规则；编辑实际文件，或先备份自己的修改，再移走对应文件后重建。
- **NameError: __file__：** 不要在普通 Notebook 单元里用它获取 Notebook 路径；本课只在真实 app.py 中使用。
- **起名冲突：** 不要把自定义文件命名为 json.py、pathlib.py 等正在使用的标准库名字，避免误导入。


## 9. 练习 A：预测直接运行与导入

不看已有输出，用自己的话写出 entry_demo.py 两种启动方式会打印什么。分别解释：顶层 print 是否执行？main 是否调用？__name__ 的值是什么？

再预测：导入后，由调用方明确写 `entry_demo.main()` 会发生什么？运行下面的留空位置，验证自己的预测。


In [ ]:
# 写下预测，再补充 run_python 调用。
# 目标代码是：import entry_demo; entry_demo.main()
# 运行目录使用前面准备好的 demo_dir。


## 10. 练习 B：增加一个可复用的统计函数

编辑实际的 project/text_tools.py，添加 `count_unknown(text, token_to_id)`，返回未知字符的出现次数。它可以先调用同一模块中的 encode，再统计编号等于 <unk> 对应值的次数。

编辑 project/app.py，在 main 中通过 `text_tools.count_unknown(...)` 调用。输入“我爱大大模型”应返回 2；“我爱学习”应返回 0；空字符串也应返回 0。

保存两个文件后，再用新进程运行。工具模块内部可以直接调用本模块的 encode；app.py 则通过 text_tools.encode 或 text_tools.count_unknown 调用。


In [ ]:
# 请编辑实际生成的两个 .py 文件并保存。
# 完成后，在这里写 run_python(["app.py"], project_dir)。
# 这里故意不自动写入文件，以便保留你的实现。


## 11. 练习 C：验证目录与入口是否符合预期

1. 从 project 目录运行 app.py，找到它打印的生成结果路径。
2. 从 project 的上一级目录运行同一个文件，比较工作目录、脚本目录与结果位置。
3. 在**可重新生成的**结果文件中保存一个明显不同的标记，例如 `{"note": "导入前的标记"}`。
4. 新进程只执行 import app，不主动调用 main，观察标记是否仍在。
5. 再直接运行 app.py，确认正常结果重新生成。

只比较两次正常输出的内容相等，不能证明“没有重写”：程序可能又写了相同内容。放入不同标记，才能直接观察导入是否触发了这次业务写入。参考答案使用独立目录，不会碰你的 project。


In [ ]:
# 完成练习 C 后，在这里记录：
# 两次启动的工作目录是否不同？脚本目录和结果位置是否相同？
# 只导入 app 后，标记是否保持？直接运行后是否恢复正常结果？


---

## 参考答案

三个答案各自包含运行辅助与所需源码，可以单独复制运行；它们重建自己的 generated_answer_* 演示目录中的指定文件，保留 project 中的学生代码。

### A：导入后主动调用 main

直接运行：先显示 __main__，然后调用 main。只导入：显示 entry_demo，不调用 main。导入后主动调用：先显示 entry_demo，再执行 main。


In [13]:
import subprocess
import sys
from pathlib import Path


def run_python(arguments, directory):
    # 用当前解释器启动一个新进程，避免沿用 Notebook 的模块缓存。
    command = [sys.executable, "-X", "utf8"] + arguments
    completed = subprocess.run(
        command,
        cwd=directory,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    print("退出码：", completed.returncode)
    if completed.returncode != 0:
        raise RuntimeError("示例进程执行失败，请查看上面的错误。")


answer_a_source = '''print("顶层 __name__：", __name__)


def main():
    print("main 被调用了")


if __name__ == "__main__":
    main()
'''

answer_a_dir = Path.cwd() / "lesson_06_workspace" / "generated_answer_a"
answer_a_dir.mkdir(parents=True, exist_ok=True)
(answer_a_dir / "entry_demo.py").write_text(answer_a_source, encoding="utf-8")
print("直接运行：")
run_python(["entry_demo.py"], answer_a_dir)
print("只导入：")
run_python(["-c", "import entry_demo"], answer_a_dir)
print("导入后主动调用：")
run_python(["-c", "import entry_demo; entry_demo.main()"], answer_a_dir)


直接运行：
顶层 __name__： __main__
main 被调用了
退出码： 0
只导入：
顶层 __name__： entry_demo
退出码： 0
导入后主动调用：
顶层 __name__： entry_demo
main 被调用了
退出码： 0


### B：在工具模块中添加函数，入口负责调用

下面用一个专用两文件目录演示答案。count_unknown 调用已有 encode，沿用同一词表规则；“大”出现两次，计数为 2。


In [14]:
import subprocess
import sys
from pathlib import Path


def run_python(arguments, directory):
    # 用当前解释器启动一个新进程，避免沿用 Notebook 的模块缓存。
    command = [sys.executable, "-X", "utf8"] + arguments
    completed = subprocess.run(
        command,
        cwd=directory,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    print("退出码：", completed.returncode)
    if completed.returncode != 0:
        raise RuntimeError("示例进程执行失败，请查看上面的错误。")


answer_b_tools = '''# 可复用的字符编码工具：导入时只定义函数。

def encode(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


def decode(token_ids, id_to_token):
    if not isinstance(token_ids, list):
        raise ValueError("token_ids 必须是列表")

    restored = ""
    for token_id in token_ids:
        if isinstance(token_id, bool):
            raise ValueError("编号不能是布尔值")
        if not isinstance(token_id, int):
            raise ValueError("编号必须是整数")
        if token_id < 0:
            raise ValueError("编号不能小于 0")
        if token_id >= len(id_to_token):
            raise ValueError("编号超出词表范围")
        restored = restored + id_to_token[token_id]
    return restored


def count_unknown(text, token_to_id):
    token_ids = encode(text, token_to_id)
    count = 0
    for token_id in token_ids:
        if token_id == token_to_id["<unk>"]:
            count = count + 1
    return count
'''

answer_b_app = '''import text_tools


def main():
    token_to_id = {
        "我": 0, "爱": 1, "学": 2, "习": 3,
        "模": 4, "型": 5, "<unk>": 6,
    }
    id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

    text = "我爱大大模型"
    print("原文：", text)
    print("编号：", text_tools.encode(text, token_to_id))
    print("未知字符数：", text_tools.count_unknown(text, token_to_id))


if __name__ == "__main__":
    main()
'''

answer_b_dir = Path.cwd() / "lesson_06_workspace" / "generated_answer_b"
answer_b_dir.mkdir(parents=True, exist_ok=True)
(answer_b_dir / "text_tools.py").write_text(answer_b_tools, encoding="utf-8")
(answer_b_dir / "app.py").write_text(answer_b_app, encoding="utf-8")
run_python(["app.py"], answer_b_dir)

zero_cases = '''import text_tools
token_to_id = {"我": 0, "爱": 1, "学": 2, "习": 3, "模": 4, "型": 5, "<unk>": 6}
print("已知文本的未知数：", text_tools.count_unknown("我爱学习", token_to_id))
print("空文本的未知数：", text_tools.count_unknown("", token_to_id))
'''

run_python(["-c", zero_cases], answer_b_dir)


原文： 我爱大大模型
编号： [0, 1, 6, 6, 4, 5]
未知字符数： 2
退出码： 0
已知文本的未知数： 0
空文本的未知数： 0
退出码： 0


### C：从不同位置启动，并用标记检查导入行为

第一次与第二次都应生成相同记录。写入标记后，只导入 app 应保留标记；最后一次直接运行则恢复正常结果。


In [15]:
import subprocess
import sys
from pathlib import Path


def run_python(arguments, directory):
    # 用当前解释器启动一个新进程，避免沿用 Notebook 的模块缓存。
    command = [sys.executable, "-X", "utf8"] + arguments
    completed = subprocess.run(
        command,
        cwd=directory,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    print(completed.stdout, end="")
    if completed.stderr:
        print(completed.stderr, end="")
    print("退出码：", completed.returncode)
    if completed.returncode != 0:
        raise RuntimeError("示例进程执行失败，请查看上面的错误。")


import json

answer_c_tools = '''# 可复用的字符编码工具：导入时只定义函数。

def encode(text, token_to_id):
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")

    token_ids = []
    for char in text:
        if char in token_to_id:
            token_id = token_to_id[char]
        else:
            token_id = token_to_id["<unk>"]
        token_ids.append(token_id)
    return token_ids


def decode(token_ids, id_to_token):
    if not isinstance(token_ids, list):
        raise ValueError("token_ids 必须是列表")

    restored = ""
    for token_id in token_ids:
        if isinstance(token_id, bool):
            raise ValueError("编号不能是布尔值")
        if not isinstance(token_id, int):
            raise ValueError("编号必须是整数")
        if token_id < 0:
            raise ValueError("编号不能小于 0")
        if token_id >= len(id_to_token):
            raise ValueError("编号超出词表范围")
        restored = restored + id_to_token[token_id]
    return restored
'''

answer_c_app = '''# 从入口调用工具模块；生成结果放在本项目目录内。
import json
from pathlib import Path
import text_tools


def main():
    project_dir = Path(__file__).resolve().parent
    output_dir = project_dir / "generated_data"
    print("当前工作目录：", Path.cwd())
    print("脚本目录：", project_dir)
    token_to_id = {
        "我": 0, "爱": 1, "学": 2, "习": 3,
        "模": 4, "型": 5, "<unk>": 6,
    }
    id_to_token = ["我", "爱", "学", "习", "模", "型", "<unk>"]

    texts = ["我爱学习", "我爱大模型", ""]
    records = []
    for text in texts:
        token_ids = text_tools.encode(text, token_to_id)
        restored = text_tools.decode(token_ids, id_to_token)
        record = {
            "text": text,
            "token_ids": token_ids,
            "decoded_text": restored,
        }
        records.append(record)

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "generated_records.json"
    json_text = json.dumps(records, ensure_ascii=False, indent=2)
    output_path.write_text(json_text, encoding="utf-8")

    print("记录数量：", len(records))
    print("结果文件：", output_path.resolve())
    for record in records:
        print(record)
    return records


if __name__ == "__main__":
    main()
'''

answer_c_dir = Path.cwd() / "lesson_06_workspace" / "generated_answer_c"
answer_c_dir.mkdir(parents=True, exist_ok=True)
(answer_c_dir / "text_tools.py").write_text(answer_c_tools, encoding="utf-8")
(answer_c_dir / "app.py").write_text(answer_c_app, encoding="utf-8")
answer_c_output = answer_c_dir / "generated_data" / "generated_records.json"

print("从项目目录启动：")
run_python(["app.py"], answer_c_dir)
first_text = answer_c_output.read_text(encoding="utf-8")

print("从项目上一级启动：")
run_python(["generated_answer_c/app.py"], answer_c_dir.parent)
second_text = answer_c_output.read_text(encoding="utf-8")
print("两次生成内容相同：", first_text == second_text)

marker = '{"note": "导入前的标记"}'
answer_c_output.write_text(marker, encoding="utf-8")
run_python(["-c", "import app; print('只导入，没有调用 main')"], answer_c_dir)
after_import = answer_c_output.read_text(encoding="utf-8")
print("导入后标记保持不变：", after_import == marker)

print("再次直接运行，恢复正常结果：")
run_python(["app.py"], answer_c_dir)
restored_text = answer_c_output.read_text(encoding="utf-8")
print("直接运行后恢复正常结果：", restored_text == first_text)
print("恢复后的记录数：", len(json.loads(restored_text)))


从项目目录启动：
当前工作目录： d:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\generated_answer_c
脚本目录： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\generated_answer_c
记录数量： 3
结果文件： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\generated_answer_c\generated_data\generated_records.json
{'text': '我爱学习', 'token_ids': [0, 1, 2, 3], 'decoded_text': '我爱学习'}
{'text': '我爱大模型', 'token_ids': [0, 1, 6, 4, 5], 'decoded_text': '我爱<unk>模型'}
{'text': '', 'token_ids': [], 'decoded_text': ''}
退出码： 0
从项目上一级启动：
当前工作目录： d:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace
脚本目录： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\generated_answer_c
记录数量： 3
结果文件： D:\code\AI_LLM_and_Deep learning\大模型学习路线\lesson_06_workspace\generated_answer_c\generated_data\generated_records.json
{'text': '我爱学习', 'token_ids': [0, 1, 2, 3], 'decoded_text': '我爱学习'}
{'text': '我爱大模型', 'token_ids': [0, 1, 6, 4, 5], 'decoded_text': '我爱<unk>模型'}
{'text': '', 'token_ids': [], 'decoded_te

## 完成本课的标准

- 能从空白文件写出一个工具函数，并从另一个文件导入使用。
- 能说明 import 会处理哪些顶层代码，以及函数体何时执行。
- 能解释 main 是普通函数，入口条件负责决定是否调用它。
- 能说明为什么导入 app 没有自动生成记录，明确调用 app.main 却会生成。
- 能完成 count_unknown 的变化题，并分别验证结果 2、0、0。
- 能区分当前工作目录、脚本所在目录和结果文件位置。
- 知道修改实际模块后保存并启动新进程，避免旧导入结果干扰观察。

可以提交练习 B 的两个 .py 源码，以及练习 C 中“标记保持不变”和“恢复正常结果”的输出。运行全部单元完成不等于独立完成这些练习。

## 与后续大模型工程的联系

后续会把数据处理、模型定义、训练入口、推理入口和 Agent 工具分开放置。今天的两文件项目是最小的练习：函数定义可被复用，启动流程有清楚的入口，文件路径有明确依据。以后增加模块数量，仍然要先考虑这些边界。

**下一课：第 07 课，自动检查与基础调试。** 用固定输入和预期输出检查函数，主动发现回归错误，再练习定位与修复。

## 官方参考资料

- [Python 3.12 教程：模块](https://docs.python.org/zh-cn/3.12/tutorial/modules.html)
- [Python 3.12：__main__ 与程序入口](https://docs.python.org/zh-cn/3.12/library/__main__.html)
- [Python 3.12：pathlib 路径操作](https://docs.python.org/zh-cn/3.12/library/pathlib.html)
- [Python 3.12：subprocess 运行辅助](https://docs.python.org/zh-cn/3.12/library/subprocess.html)
